# Phase 3: Load model + configure LoRA

**Goal:** load Phi-3 in 4-bit (to fit in GPU memory), then attach small LoRA adapter layers that are the only weights we will train.

Run after **Phase 2** (data) is ready. **Phase 4** (training) will use the `model` you build here.

## Imports

- **`AutoModelForCausalLM`:** loads the causal language model.
- **`BitsAndBytesConfig`:** enables 4-bit loading (QLoRA).
- **`LoraConfig` / `get_peft_model`:** define and attach LoRA adapters.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

## 1. Configure 4-bit quantization (QLoRA)

This shrinks the **frozen** base weights from float32 → 4-bit, cutting GPU memory roughly 4–8×. LoRA adapters stay in float16.

Look up: `load_in_4bit`, `bnb_4bit_compute_dtype`, `bnb_4bit_use_double_quant`.

Fill in `BitsAndBytesConfig(...)` below.

In [ ]:
bnb_config = BitsAndBytesConfig(
    ???
)

## 2. Load Phi-3 with quantization

- **`device_map="auto"`:** lets Accelerate place layers on GPU vs CPU.
- **`trust_remote_code=True`:** required for Phi-3 on the Hub.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

## 3. Configure LoRA

LoRA adds trainable low-rank matrices into chosen layers.

| Field | Meaning |
|--------|--------|
| **`r`** | Rank (4–16 typical; higher = more capacity, more memory) |
| **`lora_alpha`** | Scaling (often `2 * r`) |
| **`target_modules`** | Which weights to adapt; for Phi-3, `["q_proj", "v_proj"]` is a safe start. Run `model.named_modules()` to explore. |
| **`lora_dropout`** | Regularization (0.05–0.1 typical) |
| **`bias`** | Usually `"none"` for QLoRA |
| **`task_type`** | `"CAUSAL_LM"` for generative LM |

Fill in the `???` entries in `LoraConfig`.

In [ ]:
lora_config = LoraConfig(
    r=???,
    lora_alpha=???,
    target_modules=???,
    lora_dropout=???,
    bias="none",
    task_type="CAUSAL_LM",
)

## 4. Attach LoRA and inspect trainable parameters

`get_peft_model` wraps the base model so only adapter weights have `requires_grad=True`. You should see a small **trainable %** (often well under 1%).

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: e.g. "trainable params: 2,097,152 || all params: 3,823,669,248 || trainable%: 0.0548"